In [ ]:
from pathlib import Path
import uproot
import numpy as np
from tqdm import tqdm
import ROOT
from array import array

In [ ]:
SAMPLE_DIR: Path = Path("/eos/uscms/store/group/lpcdihiggsboost/sixie/analyzer/HHTo4BNtupler/ArmenVersion/nano/run3/combined")
YEARS: list[str] = ["2022", "2022EE", "2023", "2023BPix"]

In [ ]:
# xsecs
QCD_dict = {
    # "QCD-4Jets_HT-100to200": 25220000.00,
    "QCD-4Jets_HT-200to400": 1963000.00,
    "QCD-4Jets_HT-400to600": 94870.00,
    "QCD-4Jets_HT-600to800": 13420.00,
    "QCD-4Jets_HT-800to1000": 2992.00,
    "QCD-4Jets_HT-1000to1200": 879.10,
    "QCD-4Jets_HT-1200to1500": 384.50,
    "QCD-4Jets_HT-1500to2000": 125.50,
    "QCD-4Jets_HT-2000": 25.78,
}

TTbar_dict = {
    "TTtoLNu2Q": 410.30,
}

In [ ]:
pt_bins = np.array([250, 275, 300, 350, 400, 450, 500, 600, 1000])
mass_bins = np.array([50, 60, 80, 100, 120, 150, 200, 250, 300, 350])

In [ ]:
def process_samples(year, samples_dict, hist, sample_type):
        """
        Process samples and fill the histogram.
        
        Args:
            samples_dict: Dictionary of sample names to cross sections
            hist: Histogram to fill
            sample_type: String description for logging (e.g., 'QCD', 'TTbar')
        
        Returns:
            Updated histogram
        """
        for sample, xsec in tqdm(samples_dict.items(), desc=f"Processing {sample_type} samples for {year}"):
            # Construct file path
            year_dir = SAMPLE_DIR / year
            try:
                file_paths = list(year_dir.glob(f"{sample}*.root"))
                if not file_paths:
                    print(f"Warning: No files found matching {sample}*.root in {year_dir}, skipping")
                    continue
                file_path = file_paths[0]
            except Exception as e:
                print(f"Error finding file for {sample}: {e}")
                continue
            
            # Open the ROOT file
            with uproot.open(file_path) as f:
                # Get the event tree
                tree = f["tree"]
                
                # Read (mass, pt)
                fat_jet_mass = tree["fatJet1_msoftdrop"].array(library="np")
                fat_jet_pt = tree["fatJet1_pt"].array(library="np")
                event_weight = tree["weight"].array(library="np")
                
                # Get total number of events from NEvents histogram
                n_events = sum(event_weight)
                
                weights = xsec * event_weight / n_events
                
                # Fill histogram
                h, _, _ = np.histogram2d(
                    fat_jet_mass, 
                    fat_jet_pt, 
                    bins=[mass_bins, pt_bins],
                    weights=weights
                )
                
                # Add to the accumulated histogram
                hist += h
                
        
        return hist

In [ ]:
for year in YEARS:
    qcd_hist = np.zeros((len(mass_bins)-1, len(pt_bins)-1))
    ttbar_hist = np.zeros((len(mass_bins)-1, len(pt_bins)-1))

    # Process samples
    qcd_hist = process_samples(year, QCD_dict, qcd_hist, "QCD")
    ttbar_hist = process_samples(year, TTbar_dict, ttbar_hist, "TTbar")

    qcd_sum = np.sum(qcd_hist)
    ttbar_sum = np.sum(ttbar_hist)
    
    ratio_hist = np.zeros_like(qcd_hist)
    
    mask = (ttbar_hist > 0)
    if np.sum(mask) > 0:
        ratio_hist[mask] = (qcd_hist[mask] / ttbar_hist[mask]) * (ttbar_sum / qcd_sum)

    output_dir = Path("./output")
    output_dir.mkdir(exist_ok=True)

    np.save(output_dir / f"QCD_hist_{year}.npy", qcd_hist)
    np.save(output_dir / f"TTbar_hist_{year}.npy", ttbar_hist)
    np.save(output_dir / f"QCD_over_TTbar_hist_{year}.npy", ratio_hist)
    
    root_file = ROOT.TFile.Open(str(output_dir / f"QCD_over_TTbar_hist_{year}.root"), "RECREATE")
    
    # Create 2D histogram
    h_ratio = ROOT.TH2F(
        f"reweight",  # name
        f"QCD/TTbar;FatJet1_MassSD [GeV];FatJet1_pt [GeV]",  # title;x-axis;y-axis
        len(mass_bins)-1, array('d', mass_bins),  # x-axis binning
        len(pt_bins)-1, array('d', pt_bins)       # y-axis binning
    )

    # Fill histogram
    for i in range(len(mass_bins)-1):
        for j in range(len(pt_bins)-1):
            h_ratio.SetBinContent(i+1, j+1, ratio_hist[i, j])

    # Write histogram to file
    h_ratio.Write()
    root_file.Close()
    
    print(f"Saved histograms for {year} to {output_dir}")
    print(f"QCD sum: {qcd_sum:.2f}, TTbar sum: {ttbar_sum:.2f}, Ratio of sums: {ttbar_sum/qcd_sum:.6f}")